In [1]:
import sys
import torch

sys.path.append('../CMR-AI/mmaction/')
sys.path.append('../CMR-AI/')

from abc import ABCMeta, abstractmethod
from swinTransformer3D_origin import SwinTransformer3D
from models.losses.mutil_class_loss import FocalLoss, cal_auc
from models.losses.weighted_auc_f1 import get_weighted_auc_f1
from load_dataset import ACDC
from utilsss import generate_mask_matrix, pruning_mask, row_softmax
from Policy import Policy, train_agent

import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import os
from PIL import Image
import torch
from torchvision import transforms
import pandas as pd
from skimage import transform
import numpy as np
from torch import nn
import SimpleITK as sitk
from torch.utils.data import DataLoader
from scipy.ndimage import zoom
import matplotlib.pyplot as plt
from imgaug import augmenters as iaa
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

/public/home/chenweilin/.local/lib/python3.8/site-packages/albumentations/__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
TRAIN = 'train'
TEST  = 'test'
FINE_TUNE = 'fine_tune'

phase = TEST

In [3]:
## Task Split
num_class = {
        0: ['HCM', 'RV', 'DCM', 'MINF'],
        1: ['HCM', 'RV'], 
        2: ['DCM', 'MINF'],
    }

save_models = {0: 'full', 1: '1', 2: '2'}
dummy_labels = num_class[0]

models = []
for _, v in num_class.items():
    num_class_ = len(v)
    models.append(SwinTransformer3D(num_class=num_class_))
models_modules = []
for i in range(len(models)):
    models_modules.append(models[i].modules())
print(f'Total model is {len(models)}')

/usr/local/lib/python3.8/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:3214.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Total model is 3


In [4]:
## RL agent
input_size, teacher_num = 0, len(models)
for name, param in list(models[0].named_parameters())[-2:-1]:
    print(f"layer anem: {name} | size: {param.size()}")
    input_size = param.size()[-1]
agent = Policy(input_size=input_size, teacher_num=teacher_num)

layer anem: head.fc_cls.weight | size: torch.Size([4, 1024])


In [38]:
for i in range(len(models)):
    if phase == TRAIN:
        _pretrained_dict = torch.load(r'./ACDC/organmnist3d/organmnist3d_250.pth')
        pretrained_dict = {}
        for k, v in _pretrained_dict.items():
            if k.startswith('module.'):
                new_key = k.replace('module.', '')
            elif k.startswith('cls_head.'):
                new_key = k.replace('cls_head.', '')
            else:
                new_key = k 
            pretrained_dict[new_key] = v
    else:
        if phase == TEST:
            print(f'train test phase: load weightd from local file pth.')
            _pretrained_dict = torch.load(f'./ACDC/MTRL-MKD-SKD/full/epoch_200.pth')
            pretrained_dict = {}
            for k, v in _pretrained_dict.items():
                if k.startswith('module.'):
                    new_key = k.replace('module.', '')
                elif k.startswith('cls_head.'):
                    new_key = k.replace('cls_head.', '')
                else:
                    new_key = k

                pretrained_dict[new_key] = v
    model_dict = models[i].state_dict()
    pretrained_dict = {k: v for k, v in pretrained_dict.items() if k in model_dict and v.size() == model_dict[k].size()}
    model_dict.update(pretrained_dict)
    models[i].load_state_dict(pretrained_dict, strict=False)
    print(f'No.{i+1} model load pretrained weighted end.')

train test phase: load weightd from local file pth.
No.1 model load pretrained weighted end.
train test phase: load weightd from local file pth.
No.2 model load pretrained weighted end.
train test phase: load weightd from local file pth.
No.3 model load pretrained weighted end.


In [39]:
train_data = pd.read_csv('./sax_roi_processed/training/train_data.csv', encoding='GBK')
test_data = pd.read_csv('./sax_roi_processed/testing/test_data.csv', encoding='GBK')

train_data = train_data[train_data['Finding Labels'].isin(dummy_labels)]
test_data = test_data[test_data['Finding Labels'].isin(dummy_labels)]

In [40]:
# One Hot Encoding of Finding Labels to dummy_labels
for label in dummy_labels:
    train_data[label] = train_data['Finding Labels'].map(lambda result: 1.0 if label in result else 0)

In [41]:
# One Hot Encoding of Finding Labels to dummy_labels
for label in dummy_labels:
    test_data[label] = test_data['Finding Labels'].map(lambda result: 1.0 if label in result else 0)

In [42]:
train_data['target_vector'] = train_data.apply(lambda target: [target[dummy_labels].values], 1).map(lambda target: target[0])
test_data['target_vector'] = test_data.apply(lambda target: [target[dummy_labels].values], 1).map(lambda target: target[0])

In [43]:
clean_labels = train_data[dummy_labels].sum().sort_values(ascending= False) # get sorted value_count for clean labels
print(f'train data size：')
print(clean_labels) # view tabular results

train data size：
HCM     20.0
RV      20.0
DCM     20.0
MINF    20.0
dtype: float64


In [44]:
print(f'test size：')
clean_labels = test_data[dummy_labels].sum().sort_values(ascending= False) # get sorted value_count for clean labels
print(clean_labels) # view tabular results

test size：
HCM     10.0
RV      10.0
DCM     10.0
MINF    10.0
dtype: float64


### model train

In [45]:
base_lr = 0.0005
batch_size = 8
max_epoch = 600

# 将模型放到多 GPU 上
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    for i in range(len(models)):
        models[i] = nn.DataParallel(models[i])
    
for i in range(len(models)):
    models[i] = models[i].cuda()
    
fn_loss  = FocalLoss(device = 'cuda:0', gamma = 2.).to('cuda:0')
kl_loss = torch.nn.KLDivLoss(reduction='batchmean')

cross_loss = nn.CrossEntropyLoss()

optimizers = []
for i in range(len(models)):
    optimizers.append(torch.optim.SGD(models[i].parameters(), lr=base_lr))

agent_optimizer = torch.optim.SGD(agent.parameters(), lr=base_lr)
    
train_acdc_data = ACDC(data=train_data, phase = 'train', img_size=(224, 224))
train_data_loader = DataLoader(train_acdc_data, batch_size=batch_size, shuffle=True, num_workers=5)
test_acdc_data = ACDC(data=test_data, phase = 'test', img_size=(224, 224))
test_data_loader = DataLoader(test_acdc_data, batch_size=batch_size, shuffle=True, num_workers=5)

In [46]:
if 0:
    for batch_idx, (batch_data, batch_finding, batch_label) in enumerate(test_data_loader):
        # print(batch_data.shape)
        for i in range(batch_data.shape[0]):
            print(f'--------- {i} -----------')
            for j in range(batch_data.shape[2]):
                img_ = batch_data[i, 0, j, :, :].detach().numpy()
                plt.imshow(img_, cmap='gray')
                plt.show()

In [47]:
if phase == TRAIN:
    from torch.utils.tensorboard import SummaryWriter
    # 初始化 TensorBoard
    writer = SummaryWriter(log_dir='./runs/VSTMTRL-ACDC')  # 指定日志保存路径

### model eval.

In [48]:
total_acc_list = []
total_auroc_list = []

total_weight_auroc_list = []
total_weight_acc_list = []
### eval.
for i in range(10):
    test_loader_nums = len(test_data_loader.dataset)
    test_probs = np.zeros((test_loader_nums, len(dummy_labels)), dtype = np.float32)
    test_gt    = np.zeros((test_loader_nums, len(dummy_labels)), dtype = np.float32)
    test_k  =0
    models[0].eval()
    with torch.no_grad():
        for test_data_batch, _, test_label_batch in test_data_loader:
            test_data_batch = test_data_batch.cuda()
            test_label_batch = test_label_batch.cuda()
            test_outputs, _ = models[0](test_data_batch.cuda())
            test_outputs = test_outputs.reshape(test_outputs.shape[0], -1)           
            test_label_batch = test_label_batch.reshape(test_outputs.shape[0], -1)
            test_probs[test_k: test_k + test_outputs.shape[0], :] = test_outputs.cpu().detach().numpy()
            test_gt[   test_k: test_k + test_outputs.shape[0], :] = test_label_batch.cpu().detach().numpy()
            test_k += test_outputs.shape[0]
        test_label = np.argmax(test_gt, axis=1)
        test_pred = np.argmax(test_probs, axis=1)
        weight_auc, auc_list = get_weighted_auc_f1(test_probs, test_pred, test_label)

        cm = confusion_matrix(test_label, test_pred)
        dataset_list = [10, 10, 10, 10]  # , 7
        acc_list = []
        weighted_acc = 0.0
        for i in range(len(dataset_list)):
            weight = dataset_list[i] / sum(dataset_list)
            correct = cm[i][i]
            acc = float(correct) / dataset_list[i]
            acc_list.append(acc)
            weighted_acc += weight*acc 
        
        total_auroc_list.append(auc_list)
        total_acc_list.append(acc_list)
        total_weight_auroc_list.append(weight_auc)
        total_weight_acc_list.append(weighted_acc)

--------------------------------------------------
auc_list : [0.9766666666666666, 1.0, 0.98, 0.9733333333333334]
weighted_auroc:  0.9824999999999999
weighted_F1:  0.8773182957393484
--------------------------------------------------
auc_list : [0.9866666666666667, 1.0, 0.9833333333333334, 0.9700000000000001]
weighted_auroc:  0.9850000000000001
weighted_F1:  0.8746031746031747
--------------------------------------------------
auc_list : [0.9866666666666667, 1.0, 0.98, 0.9733333333333334]
weighted_auroc:  0.9850000000000001
weighted_F1:  0.8507936507936509
--------------------------------------------------
auc_list : [0.99, 1.0, 0.9766666666666667, 0.9466666666666668]
weighted_auroc:  0.9783333333333334
weighted_F1:  0.9011278195488722
--------------------------------------------------
auc_list : [0.9800000000000001, 1.0, 0.9666666666666667, 0.97]
weighted_auroc:  0.9791666666666667
weighted_F1:  0.8773182957393484


KeyboardInterrupt: 

In [ ]:
print(total_weight_auroc_list)
print(total_weight_acc_list)

In [ ]:
auc_arr = np.array(total_auroc_list)
print(auc_arr.shape)
for i in range(auc_arr.shape[-1]):
    auc_arr_cls = auc_arr[:, i]
    mean = np.mean(auc_arr_cls)
    std = np.std(auc_arr_cls)
    print(mean, std)

In [ ]:
acc_arr = np.array(total_acc_list)
print(acc_arr.shape)
for i in range(auc_arr.shape[-1]):
    acc_arr_cls = acc_arr[:, i]
    mean = np.mean(acc_arr_cls)
    std = np.std(acc_arr_cls)
    print(mean, std)

END